In [1]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /home/james/git/research/fed-learning/code
CWD: /home/james/git/research/fed-learning


In [2]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch
from tqdm import tqdm

import pandas as pd
import numpy as np

import torch.nn as nn
import torch.nn.functional as F

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

In [3]:
train_df = pd.read_csv("dataset/hm_train_df.csv")
val_df = pd.read_csv("dataset/hm_val_df.csv")
n_users = train_df['customer_id'].nunique()
n_items = train_df['product_code'].nunique()
train_df.head()

,customer_id,product_code,bought
0,0,0,2
1,0,1,2
2,0,2,1
3,0,3,1
4,0,4,1


In [4]:
X_train = train_df[["customer_id", "product_code"]]
X_train.columns = ["user_id", "item_id"]
y_train = train_df["bought"]

X_test = val_df[["customer_id", "product_code"]]
X_test.columns = ["user_id", "item_id"]
y_test = val_df["bought"]
# y_train.columns = [""]

In [1]:
class BPR(nn.Module):
    def __init__(self, n_users, n_items, n_factors, sparse=True):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors, sparse=sparse)
        self.item_factors = nn.Embedding(n_items, n_factors, sparse=sparse)
        self.item_biases = nn.Embedding(n_items, 1, sparse=sparse)
    
    def forward(self, user, pos_idx, neg_idx):
        pos_pred = (self.user_factors(user) * self.item_factors(pos_idx)).sum(-1).unsqueeze(-1) + self.item_biases(pos_idx)
        neg_pred = (self.user_factors(user) * self.item_factors(neg_idx)).sum(-1).unsqueeze(-1) + self.item_biases(neg_idx)
        return pos_pred, neg_pred

    def predict(self, user, item_idx):
        return (self.user_factors(user) * self.item_factors(pos_idx)).sum(-1).unsqueeze(-1) + self.item_biases(pos_idx)

NameError: name 'nn' is not defined

In [6]:
def random_k_out_graph(n, k, alpha, self_loops=False, seed=None):
    if alpha < 0:
        raise ValueError("alpha must be positive")
    G = nx.empty_graph(n, create_using=nx.MultiDiGraph)
    weights = Counter({v: alpha for v in G})
    for i in range(k * n):
        u = np.random.choice([v for v, d in G.out_degree() if d < k])

        if not self_loops:
            adjustment = Counter({u: weights[u]})
        else:
            adjustment = Counter()
        v = weighted_choice(weights - adjustment, seed=seed)
        G.add_edge(u, v)
        weights[v] += 1
    return G

In [7]:
#Models Initialization
def init_models():
    model_dict = {}
    for x in tqdm(range(n_users)):
        model_dict["model{0}".format(x)] = BPR(n_users,n_items, n_factors=30, sparse=True)
    return model_dict

#Optimizer selection
def get_optimizer(model, sparse=False):
    if sparse:
        return torch.optim.SparseAdam(model.parameters())
    else:
        return torch.optim.SGD(model.parameters(), lr=0.01)


def validate(model_dict, val_loader):

    tbar = val_loader
    criterion = nn.MSELoss()
    loss_list = []
    for i in range(len(model_dict)):
        model_dict["model{0}".format(i)].eval()

    with torch.no_grad():
        for idx, (inputs, target) in enumerate(tbar):
            logits = model_dict["model{0}".format(int(inputs[:, 0]))](inputs[:,0], inputs[:,1])
            loss = criterion(logits, target.float())

            loss_list.append(loss.detach().cpu().item())
        avg_loss = np.sqrt(np.mean(loss_list))

    return avg_loss

def train(model_dict, train_loader, epochs, val_loader, graph, sparse=True):

    criterion = nn.MSELoss()
    val_loss = []
    val_map = validate(model_dict, val_loader)
    val_loss.append(val_map)

    #Initialize Optimizers
    optimizer_dict = {}
    for i in tqdm(range(n_users)):
        optimizer_dict["model{0}".format(i)] = get_optimizer(model_dict["model{0}".format(i)], sparse=sparse)

    for e in range(epochs):
        tbar = tqdm(train_loader, file=sys.stdout)
        for i in range(len(model_dict)):
            model_dict["model{0}".format(i)].train()
        # loss_list = []
        losses = np.empty(len(train_loader))
        for idx, (inputs, target) in enumerate(tbar):
            # inputs, target = read_data(data)
            # # inputs = inputs[0]
            # print(f"{inputs.shape=}")
            # print(f"{target.shape=}")
            optimizer = optimizer_dict["model{0}".format(int(inputs[:, 0]))]
            optimizer.zero_grad()
            logits = model_dict["model{0}".format(int(inputs[0, 0]))](inputs[:,0], inputs[:, 1])
            loss = criterion(logits, target.float())
            loss.backward()#Calculate Gradients

            #Get gradients of current Model
            model_gradient = {}#Store current gradients
            for name,param in model_dict["model{0}".format(int(inputs[:, 0]))].named_parameters():
                gradient = param.grad
                model_gradient[name] = gradient

            #Assign gradients to neighbors' Model
            user_n = graph.adj[int(inputs[:, 0])]
            # user_n = list(Adjacency_List[int(inputs[0])])
            #user_n = []  #uncomment it for local learning
            for neighbor in user_n:
                neighbor_model = model_dict["model{0}".format(neighbor)]
                neighbor_optimizer = optimizer_dict["model{0}".format(neighbor)]
                neighbor_model.zero_grad()
                for name,param in neighbor_model.named_parameters():
                    param.grad = model_gradient[name] #Assign gradients to the neighbors
                neighbor_optimizer.step()#Neighbors' update

            optimizer.step()#Current user's update
            losses[idx] = loss.detach().numpy()
            # loss_list.append(loss.detach().cpu().item())
            avg_loss = losses[:(idx+1)].mean()
            tbar.set_description(f"Epoch {e+1} Loss: {np.sqrt(np.round(avg_loss,7))} ")

        val_map = validate(model_dict, val_loader)
        val_loss.append(val_map)
        log_text = f"Epoch {e+1}\nTrain Loss: {np.sqrt(avg_loss)}\nValidation Loss: {val_map}\n"
        print(log_text)

In [8]:
# train_dataset = HMDataset(X_train,y_train)
graph = random_k_out_graph(n_users,5,50)
train_dataset = TensorDataset(torch.tensor(X_train.to_numpy(), dtype=int), torch.tensor(y_train.to_numpy(), dtype=float))
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_dataset = TensorDataset(torch.tensor(X_test.to_numpy(), dtype=int), torch.tensor(y_test.to_numpy(), dtype=float))
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)
model_dict = init_models()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1760/1760 [00:05<00:00, 337.81it/s]


In [9]:
train(model_dict, train_loader,epochs=50, val_loader=val_loader, graph=graph, sparse=True)

TypeError: BPR.forward() missing 1 required positional argument: 'neg_idx'